# LSTM(Long Short-Term Memory)
- **순환신경망(RNN, Recurrent Neural Network)** 의 한 종류로, 장기 의존성(long-term dependency) 문제를 해결하기 위해 고안된 신경망 구조이다.
- 기존 RNN은 **장기 기억이 어려운 문제(vanishing gradient)** 가 존재
- LSTM은 셀 상태(cell state)와 게이트 구조를 활용하여 장기 의존성을 효과적으로 학습 가능


- 주요 특징장기
    - 기억을 유지할 수 있음
    - 그래디언트 소실(vanishing gradient) 문제 해결
    - 게이트 구조(입력, 출력, 망각 게이트)를 통해 중요한 정보만 유지
    - 자연어 처리(NLP), 시계열 데이터 예측 등에 활용됨

## 1.1.주식 데이터 가져오기
- yfinance에서 데이터를 불러올 때, Ticker(티커)라는 것을 인자로 넣어준다.
- Ticker는 야후 파이낸스 홈페이지에 원하는 종목을 검색하면 Ticker를 볼 수 있다.
- https://finance.yahoo.com/
- Ticker 목록 데이터 : https://www.nasdaq.com/market-activity/stocks/screener?page=1&rows_per_page=25

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

## 1.2.LSTM으로 주식 예측하기

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# 종가만 사용


In [ ]:
# 데이터 스케일링


In [ ]:
# 시퀀스 데이터 생성
sequence_length =
X, y = [], []

for i in range(len(scaled_data) - sequence_length):
    X.append(scaled_data[i:i+sequence_length])
    y.append(scaled_data[i+sequence_length])

X, y = np.array(X), np.array(y)

In [ ]:
# 데이터 분할
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [ ]:
# 텐서 변환
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

In [ ]:
# 데이터 로더 생성
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)


- 모델 정의

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_layer_size=50, output_size=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_layer_size, batch_first=True)
        self.linear = nn.Linear(hidden_layer_size, output_size)

    # 순전파 연산
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.linear(lstm_out[:,-1,:])

- 모델 학습

In [ ]:
losses = []

# 모델 학습
model.train()
epochs = 30

for epoch in range(epochs):
    epoch_loss = 0  # 한 epoch 동안의 총 loss 저장
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)       # 손실 계산
        loss.backward()         # 역전파 수행
        optimizer.step()        # 가중치 업데이트
        epoch_loss += loss.item()       # 현재 배치의 손실값을 epoch_loss에 추가

    # 평균 loss
    avg_loss = epoch_loss / len(train_loader)       # 배치 개수로 나누어 평균 손실 계산
    losses.append(avg_loss)

    print(f'Epoch {epoch+1}/{epochs}, Loss : {avg_loss:.4f}')

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(range(1, epochs + 1), losses, marker='o', linestyle='-', color='b', label='Traning Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training loss over epochs')
plt.legend()
plt.grid()
plt.show()

- 예측 수행

- 결과 시각화

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(real_prices, label='Actual Price')
plt.plot(predicted_prices, label='predicted prices')
plt.xlabel('Time')
plt.ylabel('Price')
plt.title('Stock price prediction')
plt.legend()
plt.grid()
plt.show()

- 주식 예측

In [ ]:
with torch.no_grad():
    for _ in range(future_days):
        seq_tensor = torch.tensor(recent_sequence[-sequence_length:], dtype=torch.float32).unsqueeze(0).to(device)
        pred = model(seq_tensor).cpu().numpy()
        recent_sequence = np.append(recent_sequence, pred, axis=0)
        predictions.append(pred[0, 0])

# 스케일 역변환
predicted_future_prices = scaler.inverse_transform(np.array(predictions).reshape(-1, 1))

# 1차원 배열로 변환
predicted_future_prices = predicted_future_prices.flatten()

In [ ]:
# 시각화
plt.figure(figsize=(12, 6))
plt.plot(real_prices, label='Actual Price')

# x축 인덱스 설정
plt.plot(range(len(real_prices), len(real_prices) + future_days),
         predicted_future_prices, label='Future Predicted Price', linestyle='--')

plt.title('Future Stock Price Prediction using PyTorch LSTM')
plt.xlabel('Time (Days)')
plt.ylabel('Price')
plt.legend()
plt.show()